In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/raw/diabetes.csv')

# Inspect structure
print("Dataset Shape:", df.shape)
print("\nData Types & Null Values:")
print(df.info())
print("\nFirst 5 Rows:")
df.head()

Dataset Shape: (768, 9)

Data Types & Null Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    float64
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    float64
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(4), int64(5)
memory usage: 54.1 KB
None

First 5 Rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [3]:
print("Outcome counts:")
print(df['Outcome'].value_counts())
print("\nOutcome proportions:")
print(df['Outcome'].value_counts(normalize=True))

Outcome counts:
Outcome
0    500
1    268
Name: count, dtype: int64

Outcome proportions:
Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64


In [4]:
print("\nMissing values per column:")
print(df.isnull().sum())

suspect_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in suspect_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count} zero values")


Missing values per column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64
Glucose: 0 zero values
BloodPressure: 0 zero values
SkinThickness: 0 zero values
Insulin: 0 zero values
BMI: 0 zero values


In [5]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Outcome'])
y = df['Outcome']


feature_order = list(X.columns)
print("Features List:", feature_order)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain shape: {X_train_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")

Features List: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Train shape: (614, 8)
Test shape: (154, 8)


In [7]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)

print("--- Random Forest Evaluation ---")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, rf_preds))

# 2. Train XGBoost
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train_scaled, y_train)
xgb_preds = xgb.predict(X_test_scaled)

print("\n--- XGBoost Evaluation ---")
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, xgb_preds))

--- Random Forest Evaluation ---
Accuracy: 0.8636363636363636
ROC-AUC: 0.9447222222222221
              precision    recall  f1-score   support

           0       0.89      0.90      0.90       100
           1       0.81      0.80      0.80        54

    accuracy                           0.86       154
   macro avg       0.85      0.85      0.85       154
weighted avg       0.86      0.86      0.86       154


--- XGBoost Evaluation ---
Accuracy: 0.8896103896103896
ROC-AUC: 0.9427777777777777
              precision    recall  f1-score   support

           0       0.92      0.91      0.91       100
           1       0.84      0.85      0.84        54

    accuracy                           0.89       154
   macro avg       0.88      0.88      0.88       154
weighted avg       0.89      0.89      0.89       154



In [9]:
from sklearn.metrics import recall_score, roc_auc_score

rf_recall = recall_score(y_test, rf_preds, pos_label=1)
xgb_recall = recall_score(y_test, xgb_preds, pos_label=1)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:, 1])
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test_scaled)[:, 1])

print(f"Random Forest  -> Recall (Class 1): {rf_recall:.4f} | ROC-AUC: {rf_auc:.4f}")
print(f"XGBoost        -> Recall (Class 1): {xgb_recall:.4f} | ROC-AUC: {xgb_auc:.4f}")

# Decision rule
if xgb_recall > rf_recall or (xgb_recall == rf_recall and xgb_auc >= rf_auc):
    print("\nBest Model: XGBoost")
    best_model = xgb
else:
    print("\nBest Model: Random Forest")
    best_model = rf

Random Forest  -> Recall (Class 1): 0.7963 | ROC-AUC: 0.9447
XGBoost        -> Recall (Class 1): 0.8519 | ROC-AUC: 0.9428

Best Model: XGBoost


In [10]:
import os
import joblib

output_dir = '../app/modules/diabetes'
os.makedirs(output_dir, exist_ok=True)


joblib.dump(xgb, os.path.join(output_dir, 'model.pkl'))
joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))

with open(os.path.join(output_dir, 'feature_order.json'), 'w') as f:
    json.dump(feature_order, f)

print("Saved model.pkl, scaler.pkl, and feature_order.json to app/modules/diabetes/")

Saved model.pkl, scaler.pkl, and feature_order.json to app/modules/diabetes/
